# 04 — Comparative Analysis
Anomaly overlap, neighborhood Jaccard, cross-dataset.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

Setup done.


## Load Saved Kernels & Scores

In [2]:
from src.data import load_wine_quality, scale_features_to_pi, get_deterministic_sample, WINE_QUANTUM_FEATURES, WINE_TARGET_COL
from sklearn.preprocessing import StandardScaler
from src.classical import run_isolation_forest

df = load_wine_quality('../data/raw/winequality-red.csv')
N = 150
idx, df_sample = get_deterministic_sample(df, N, seed=42)

K_q = np.load('../outputs/kernels/K_quantum.npy')
K_c = np.load('../outputs/kernels/K_classical.npy')

X_q4 = df_sample[WINE_QUANTUM_FEATURES].values
X_q4_std = StandardScaler().fit_transform(X_q4)
CAS = run_isolation_forest(X_q4_std, random_state=42)
quality_sample = df_sample[WINE_TARGET_COL].values
print(f"K_q: {K_q.shape}, K_c: {K_c.shape}")

Loaded wine quality: 1599 rows × 12 columns
K_q: (150, 150), K_c: (150, 150)


## Quantum Anomaly Score

In [3]:
from src.interestingness import quantum_anomaly_score
QAS = quantum_anomaly_score(K_q)
print(f"QAS: mean={QAS.mean():.4f}, max={QAS.max():.4f}")

QAS: mean=0.6128, max=1.0000


## Anomaly Category Labels

In [4]:
from src.metrics import anomaly_category_labels, top_k_overlap
labels = anomaly_category_labels(CAS, QAS, pct=90)
cats, counts = np.unique(labels, return_counts=True)
for cat, cnt in zip(cats, counts):
    print(f"  {cat}: {cnt}")

  Both: 2
  Classical-only: 13
  Neither: 122
  Quantum-only: 13


## Classical vs Quantum Anomaly Scatter

In [5]:
cat_colors = {'Both':'purple','Classical-only':'blue','Quantum-only':'red','Neither':'gray'}
fig, ax = plt.subplots(figsize=(9,7))
for cat in ['Neither','Classical-only','Quantum-only','Both']:
    mask = labels == cat
    ax.scatter(CAS[mask], QAS[mask], c=cat_colors[cat], label=cat, alpha=0.7, s=25, edgecolors='none')
ax.axvline(np.percentile(CAS,90), color='blue', ls='--', lw=1, label='Classical 90th pct')
ax.axhline(np.percentile(QAS,90), color='red', ls='--', lw=1, label='Quantum 90th pct')
ax.set_xlabel('Classical Anomaly Score (IsolationForest)')
ax.set_ylabel('Quantum Anomaly Score (QAS)')
ax.set_title('Classical vs Quantum Anomaly Scores (N=150)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/08_anomaly_scatter.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 08_anomaly_scatter.png")

Saved 08_anomaly_scatter.png


## Top-k Anomaly Overlap

In [6]:
overlap_rows = []
for k in [5, 10, 15, 20, 30]:
    ov = top_k_overlap(CAS, QAS, k)
    overlap_rows.append({'k': k, 'overlap': round(ov, 4)})
    print(f"  Top-{k} overlap: {ov:.4f}")
pd.DataFrame(overlap_rows).to_csv('../outputs/tables/anomaly_overlap.csv', index=False)
print("Saved anomaly_overlap.csv")

  Top-5 overlap: 0.0000
  Top-10 overlap: 0.2000
  Top-15 overlap: 0.1333
  Top-20 overlap: 0.2000
  Top-30 overlap: 0.2667
Saved anomaly_overlap.csv


## Neighborhood Jaccard Overlap

In [7]:
from src.metrics import neighborhood_jaccard_summary
print("Computing neighborhood Jaccard overlaps...")
jac_results = neighborhood_jaccard_summary(X_q4_std, K_q, k_values=[5,10,15])
rows = []
for k, res in jac_results.items():
    rows.append({'k': k, 'mean_jaccard': round(res['mean'],4),
                 'median_jaccard': round(res['median'],4), 'std_jaccard': round(res['std'],4)})
pd.DataFrame(rows).to_csv('../outputs/tables/neighborhood_overlap.csv', index=False)
print("Saved neighborhood_overlap.csv")

Computing neighborhood Jaccard overlaps...
  Jaccard k=5: mean=0.1064, median=0.1111, std=0.1184
  Jaccard k=10: mean=0.1004, median=0.1111, std=0.0799
  Jaccard k=15: mean=0.1085, median=0.1111, std=0.0828
Saved neighborhood_overlap.csv


## Cross-Dataset: sklearn Wine

In [ ]:
from sklearn.datasets import load_wine
from src.quantum import build_zz_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.metrics import centered_kernel_alignment, top_k_overlap, neighborhood_jaccard_summary
from src.interestingness import quantum_anomaly_score

bunch = load_wine()
df_wine2 = pd.DataFrame(bunch.data, columns=bunch.feature_names)
variances = df_wine2.var().sort_values(ascending=False)
top4_feat = variances.head(4).index.tolist()
print(f"sklearn wine top-4 features by variance: {top4_feat}")

X2 = df_wine2[top4_feat].values
X2_std = StandardScaler().fit_transform(X2)
from src.data import scale_features_to_pi
X2_scaled = scale_features_to_pi(X2)

print("Computing quantum kernel for sklearn wine (N=150 sample)...")
rng = np.random.RandomState(42)
idx2 = np.sort(rng.choice(len(X2_scaled), 150, replace=False))
X2_sub = X2_scaled[idx2]
X2_std_sub = X2_std[idx2]

fm2 = build_zz_feature_map(4, reps=2)
qk2 = build_quantum_kernel(fm2)
K_q2, elapsed2 = compute_kernel_matrix(qk2, X2_sub)

from src.classical import compute_classical_kernel
K_c2 = compute_classical_kernel(X2_std_sub)
cka2 = centered_kernel_alignment(K_q2, K_c2)
QAS2 = quantum_anomaly_score(K_q2)
CAS2 = run_isolation_forest(X2_std_sub, random_state=42)
ov5_2 = top_k_overlap(CAS2, QAS2, 5)
ov10_2 = top_k_overlap(CAS2, QAS2, 10)
jac_res2 = neighborhood_jaccard_summary(X2_std_sub, K_q2, k_values=[5,10])

ov5_1 = top_k_overlap(CAS, QAS, 5)
ov10_1 = top_k_overlap(CAS, QAS, 10)
jac_res1 = neighborhood_jaccard_summary(X_q4_std, K_q, k_values=[5,10])

cross_rows = [
    {'dataset':'Red Wine Quality','sample_size':150,'qubits':4,'feature_map':'ZZFeatureMap','reps':2,
     'kernel_alignment':round(float(np.load("../outputs/kernels/K_quantum.npy") is not None and
         centered_kernel_alignment(np.load("../outputs/kernels/K_quantum.npy"),
                                   np.load("../outputs/kernels/K_classical.npy")),4),
     'mean_neighborhood_jaccard':round(jac_res1[10]["mean"],4),
     'top5_anomaly_overlap':round(ov5_1,4),'top10_anomaly_overlap':round(ov10_1,4),
     'n_quantum_only':int((labels=="Quantum-only").sum())},
    {'dataset':'sklearn Wine','sample_size':150,'qubits':4,'feature_map':'ZZFeatureMap','reps':2,
     'kernel_alignment':round(cka2,4),
     'mean_neighborhood_jaccard':round(jac_res2[10]["mean"],4),
     'top5_anomaly_overlap':round(ov5_2,4),'top10_anomaly_overlap':round(ov10_2,4),
     'n_quantum_only':int((anomaly_category_labels(CAS2, QAS2)=="Quantum-only").sum())}
]
pd.DataFrame(cross_rows).to_csv('../outputs/tables/cross_dataset_results.csv', index=False)
np.save('../outputs/kernels/K_quantum2.npy', K_q2)
np.save('../outputs/kernels/K_classical2.npy', K_c2)
print("Saved cross_dataset_results.csv")
print("Notebook 04 complete.")

SyntaxError: closing parenthesis '}' does not match opening parenthesis '(' on line 42 (52665100.py, line 47)